# 08.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/04-backpropagation-by-hand/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

def sigmoid(z): return 1 / (1 + np.exp(-z))

x = np.array([1.0, 0.5])            # one customer, two scaled features
y = 1.0                             # they churned
W = np.array([[0.2, 0.6],           # W[i, j]: input i → hidden neuron j
              [-0.4, 0.1]])
b = np.array([0.0, 0.0])            # hidden biases
v = np.array([0.5, -0.3])           # hidden → output weights
c = 0.0                             # output bias
lr = 0.5

def forward(x, W, b, v, c):
    h = sigmoid(x @ W + b)
    y_hat = sigmoid(h @ v + c)
    return h, y_hat

def backward(x, y, W, b, v, c):
    """Gradients of the log loss for one example, exactly as in the README table."""
    h, y_hat = forward(x, W, b, v, c)
    d_out = y_hat - y                       # dL/dz_out
    dv, dc = d_out * h, d_out               # output weights and bias
    dh = d_out * v                          # dL/dh_j
    d_hidden = dh * h * (1 - h)             # dL/dz_j
    dW = np.outer(x, d_hidden)              # dW[i, j] = x_i * delta_j
    return dW, d_hidden, dv, dc

def log_loss(y, y_hat):
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

def train_xor(seed, steps=1000, lr=2.0, hidden=2):
    """Full-batch gradient descent on the 4 XOR cases, averaging gradients over them."""
    rng = np.random.default_rng(seed)
    W, b = rng.normal(0, 1, (2, hidden)), np.zeros(hidden)
    v, c = rng.normal(0, 1, hidden), 0.0
    losses = []
    for _ in range(steps):
        grads = [backward(xi, yi, W, b, v, c) for xi, yi in zip(X_xor, y_xor)]
        W = W - lr * np.mean([g[0] for g in grads], axis=0); b = b - lr * np.mean([g[1] for g in grads], axis=0)
        v = v - lr * np.mean([g[2] for g in grads], axis=0); c = c - lr * np.mean([g[3] for g in grads])
        losses.append(np.mean([log_loss(yi, forward(xi, W, b, v, c)[1]) for xi, yi in zip(X_xor, y_xor)]))
    return (W, b, v, c), losses

## Exercise 1: forward pass with the updated weights

By hand, then in code: use the updated weights from the README table and confirm $\hat y = 0.6163$ and $L = 0.484$.

In [ ]:
# your code here

## Exercise 2: keep going

Take 50 update steps on the same single customer with `backward`. Plot the loss. Does it keep falling?

In [ ]:
# your code here

## Exercise 3: a customer who stayed

Repeat the backward pass with $y = 0$ (same inputs and starting weights). Which weights move in the opposite direction?

In [ ]:
# your code here

## Exercise 4: rescue seed 5

`train_xor(seed=5)` stalls. Try (a) `steps=5000`, (b) `lr=5.0`, (c) `hidden=4`. Which fixes it?

In [ ]:
# your code here